In [ ]:
import requests
import pandas as pd

In [ ]:
import sys
sys.path.append("..")

from API_Key import gnews_API

In [ ]:
# added a few more queries than before so we actually pull enough volume
queries = [
    "SAP",
    "SAP AI",
    "SAP ERP",
    "SAP cloud",
    "SAP S/4HANA",
    "SAP earnings",
    "SAP partnership",
    "enterprise software",
    "business AI",
    "SAP Technologies",
    "SAP competitors",
    "ERP market"
]

In [ ]:
API_KEY = gnews_API

In [ ]:
import time

In [ ]:
all_articles = []

for q in queries:

    url = (
        f"https://gnews.io/api/v4/search?"
        f"q={q}&lang=en&max=10&apikey={API_KEY}"
    )

    response = requests.get(url)
    time.sleep(0.2)
    data = response.json()

    if "articles" in data:
        all_articles.extend(data["articles"])
    else:
        print(f"Query failed: {q}")
        print(data)

In [ ]:
len(all_articles)

In [ ]:
gnews_docs = []

for article in all_articles:

    gnews_docs.append({
        "title": article["title"],
        "description": article["description"],
        "content": article["content"],
        "source": article["source"]["name"],
        "published": article["publishedAt"],
        "url": article["url"]
    })


In [ ]:
gnews_df = pd.DataFrame(gnews_docs)

len(gnews_df)

Removing Duplicates

In [ ]:
print("Before:", len(gnews_df))

gnews_df = gnews_df.drop_duplicates(
    subset=["title"]
)

print("After:", len(gnews_df))

In [ ]:
gnews_df.head(1)

In [ ]:
from rapidfuzz import fuzz

titles = gnews_df["title"].fillna("").tolist()

to_drop = set()

print("Before:", len(gnews_df))
for i in range(len(titles)):
    if i in to_drop:
        continue

    for j in range(i + 1, len(titles)):
        score = fuzz.ratio(titles[i], titles[j])

        if score >= 80:
            to_drop.add(j)

gnews_df = gnews_df.drop(index=list(to_drop)).reset_index(drop=True)

print("After :", len(gnews_df))

In [ ]:
gnews_df.to_json(
    "../data/gnews_articles.json",
    orient="records",
    indent=2
)